# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

### Task 1 — Results and Answers

**Environment setup:** Three isolated virtual environments were created as required:
- `speculators_venv` — Speculators v0.5.0 (editable install) for data preparation and EAGLE-3 training
- `vllm_venv` — vLLM v0.20.0 with fastapi<0.137 for serving and benchmarking
- `comp_venv` — llmcompressor v0.12.0 for FP8 dynamic quantization

**Data preparation completed:**
- Model: `Qwen/Qwen3-8B`
- Dataset: ShareGPT-style conversations
- Samples prepared: 3,000
- Sequence length: 2,048
- vLLM max context: `SEQ_LENGTH + 64 = 2112` (patched to avoid off-by-one with hidden-state generation requests)
- Total disk usage after hidden-state generation: ~122 GB
- Hidden-state files: 3,000 (zero empty files)
- GPU: NVIDIA H100 80GB HBM3, CUDA 13.0, Driver 580.159.04

**Artifacts uploaded to:**
- Hugging Face: `nodozi/eagle3-qwen3-8b-sharegpt`
- Weights & Biases: entity `team-ave`, project `spec-dec-quant-hw`

---

**Question: Why do hidden states require much more disk space than the original text dataset?**

The original text dataset stores token IDs — small integers, typically 4 bytes each. For a 2,048-token sequence, that is roughly 8 KB per sample.

Hidden states store the full intermediate activations from every transformer layer. For Qwen3-8B, each layer produces a dense float vector of dimension 4,096 for every token position. With ~36 layers and 2,048 positions, a single sample stores approximately `36 × 2,048 × 4,096 × 2 bytes (BF16) ≈ 576 MB` of activation data. Even with some compression or subset of layers, this dwarfs the text representation by orders of magnitude.

In concrete terms: 3,000 text samples might be ~25 MB, while 3,000 hidden-state samples consumed ~122 GB — roughly a 5,000× expansion. This is the fundamental cost of offline EAGLE-3 training: precomputing hidden states trades GPU time for disk space, enabling single-GPU sequential training at the expense of large storage requirements.

# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Reference training result from one run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.509` |
| `val/full_acc_0_epoch` | `0.463` |
| `val/cond_acc_0_epoch` | `0.463` |
| `val/loss_1_epoch` | `3.778` |
| `val/full_acc_1_epoch` | `0.181` |
| `val/cond_acc_1_epoch` | `0.364` |
| `val/loss_2_epoch` | `4.550` |
| `val/full_acc_2_epoch` | `0.069` |
| `val/cond_acc_2_epoch` | `0.320` |
| `val/loss_epoch` | `10.837` |
| Epoch | `4` |

Note: Epoch=4 means this is the 5th epoch, the index starts with 0.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

---

### Task 2 — Results and Answers

**Training completed successfully.** Best checkpoint: epoch 4 (5th epoch, 0-indexed).

**Training configuration:**
- Verifier model: `Qwen/Qwen3-8B`
- Training data: 3,000 samples from prepared hidden states
- Checkpoint directory: `outputs/checkpoints/eagle3_qwen3_8b_sharegpt_full_20260701_110123/`
- Best checkpoint symlink: `checkpoint_best → 4`
- Best checkpoint size: ~2 GB (model.safetensors) + ~1 GB (optimizer state)

**Our validation metrics vs. reference:**

| Metric | Our Result | Reference |
| --- | ---: | ---: |
| `val/loss_epoch` | `10.861` | `10.837` |
| `val/full_acc_0_epoch` | `0.464` | `0.463` |
| `val/full_acc_1_epoch` | `0.182` | `0.181` |
| `val/full_acc_2_epoch` | `0.069` | `0.069` |
| Best epoch | `4` | `4` |

Results closely match the reference, confirming correct data preparation and training.

---

**Question 1: What do `full_acc` and `cond_acc` measure in speculative decoding training?**

- **`full_acc_k`** (full accuracy at position k): The probability that *all* draft tokens from position 0 through position k are correct. It is a joint accuracy — every token in the chain up to and including position k must match the verifier's output. For position 0, `full_acc_0 = cond_acc_0` since there are no preceding positions.

- **`cond_acc_k`** (conditional accuracy at position k): The probability that the draft token at position k is correct, *given that all previous positions (0 through k−1) were already correct*. This isolates how well the draft model predicts at that specific position, conditioned on being on the right trajectory.

The relationship: `full_acc_k = cond_acc_0 × cond_acc_1 × ... × cond_acc_k`. This is visible in our results: `full_acc_1 (0.182) ≈ full_acc_0 (0.464) × cond_acc_1 (0.364) ≈ 0.169` — roughly consistent, with some variance from finite evaluation samples.

**Question 2: Why does accuracy usually decrease for later speculative positions?**

Each speculative position compounds uncertainty. The draft model at position k must predict what the verifier would output *k steps into the future*, without access to the verifier's actual intermediate hidden states for those steps. Three factors drive the decrease:

1. **Error accumulation:** The draft model auto-regressively feeds its own predictions forward. Any error at position k poisons positions k+1, k+2, etc. The `full_acc` metric reflects this compounding — it's a product of conditional accuracies, so it shrinks geometrically.

2. **Distribution shift:** At later positions, the draft model operates on its own generated context, which diverges from the verifier's distribution. The further ahead it speculates, the more its input distribution drifts from what it was trained on.

3. **Intrinsic prediction difficulty:** Predicting more tokens into the future is fundamentally harder. Language has branching possibilities — even a perfect model of the verifier faces genuine uncertainty about which path the verifier would take at later positions.

This is why `cond_acc` also drops (0.463 → 0.364 → 0.320): even conditioned on being correct so far, each additional step is harder.

**Question 3: What would you change if the first-position accuracy is very low?**

Low `full_acc_0` signals a problem *upstream* of the training recipe, because position 0 is the easiest prediction. Investigation order:

1. **Inspect the data generation pipeline first.** Verify that hidden states were generated by the correct model (`Qwen/Qwen3-8B`) at the correct vLLM version. A version mismatch between the hidden-state generator and the training code can produce misaligned features. Check that sequence lengths match between tokenized inputs and hidden-state files.

2. **Verify data integrity.** Check for zero-byte hidden-state files, truncated sequences, or stale `/tmp/hidden_states/` artifacts from previous failed runs contaminating the dataset.

3. **Increase training data.** If the pipeline is correct but accuracy is low, 3,000 samples may be insufficient. Increasing to 5,000–10,000 samples gives the draft head more diverse examples of the verifier's behavior.

4. **Only then adjust training hyperparameters** — learning rate, number of epochs, batch size. But these are secondary to data quality for position-0 accuracy, since the draft head at position 0 has the most information (the actual verifier hidden state as input) and should learn the mapping fairly easily if the data is correct.

## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

---

### Task 3 — Results and Answers

**Quantization completed successfully.**
- Tool: `llmcompressor` v0.12.0
- Method: FP8 dynamic quantization (W8A8) via compressed tensors
- Target modules: all linear layers except `lm_head`
- Saved to: `models/Qwen3-8B-FP8-Dynamic` (separate from original BF16 model)

**FP8 benchmark result:** 1618.27 tok/s output throughput (1.43× baseline), passing the 1550 tok/s threshold.

---

**Question 1: Why is FP8 dynamic quantization useful for serving on H100?**

H100 has dedicated FP8 Tensor Cores that deliver 2× the FLOPS of BF16 (3,958 vs 1,979 TFLOPS). FP8 dynamic quantization exploits this by storing weights in 8-bit format and computing activations in FP8 at runtime. This halves memory bandwidth consumption per token — the primary bottleneck during autoregressive decoding, which is memory-bound rather than compute-bound. In our benchmark, this translated to TPOT dropping from 7.01 ms to 4.87 ms (30% reduction), directly reflecting the bandwidth savings. Dynamic quantization requires no calibration dataset, making it straightforward to apply.

**Question 2: Why might `lm_head` be excluded from quantization?**

The `lm_head` is the final projection layer that converts hidden states into logits over the entire vocabulary (~150K tokens for Qwen3). Small numerical errors here get amplified across the softmax distribution, distorting token selection probabilities. Unlike intermediate layers where errors are absorbed by subsequent transformations, `lm_head` errors directly change which token gets selected. The layer is also relatively small compared to the transformer stack, so excluding it has negligible impact on memory savings while preserving output quality.

**Question 3: How can quantization affect speculative decoding acceptance rate?**

Quantization changes the verifier's output distribution slightly. The draft head was trained to predict the BF16 verifier's hidden states and next-token decisions. When the verifier switches to FP8, its logits shift — some tokens that were marginal become more or less likely. This distribution mismatch can either help or hurt acceptance rate depending on the specific model and data. In the reference results, FP8+speculative actually showed higher acceptance (36.50% vs 22.48%), suggesting the FP8 verifier's distribution became slightly more predictable for the draft head. In our results, acceptance stayed similar (~15% for nspec=3, ~36% for nspec=1), indicating the quantization effect was neutral.

## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

Important:

The reference results used different numbers of speculative draft tokens for the unquantized speculative-decoding run and the FP8 + speculative-decoding run. Do not assume one value is optimal for both. Tune it yourself and justify the final choice using throughput, acceptance rate, acceptance length, and TPOT.

Hints:

- Start with a small number of speculative tokens, then increase only if the acceptance behavior supports it.
- Compare output token throughput first, then use TTFT, TPOT, and acceptance metrics to explain the result.
- If a setting produces more draft work but little accepted work, it is probably not the best setting.

Script for benchmarking:

```bash
vllm bench serve \
    --model Qwen/Qwen3-8B \
    --dataset-name hf \
    --max-concurrency 8 \
    --dataset-path philschmid/mt-bench \
    --num-prompts 80
```


Reference benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `24.35` | `3.29` | `841.22` | `1090.87` | `576.17` | `7.28` | N/A |
| Speculative decoding | `16.27` | `4.92` | `1258.65` | `1632.19` | `78.17` | `5.76` | `22.48%` |
| FP8 quantization | `13.06` | `6.12` | `1566.56` | `2031.82` | `51.18` | `4.90` | N/A |
| FP8 + speculative decoding | `11.59` | `6.90` | `1766.55` | `2290.82` | `30.24` | `4.28` | `36.50%` |

Reference speculative decoding details:

| Configuration | Reference draft tokens | Acceptance length | Drafts | Draft tokens | Accepted tokens |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative decoding | `2` | `1.45` | `14088` | `28176` | `6334` |
| FP8 + speculative decoding | `1` | `1.36` | `14954` | `14954` | `5458` |

Your exact numbers may differ, but the relative comparison should be explainable.

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Points |
| --- | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | 15 |
| **Total** |  | **50** |




### Task 4 — Results and Answers

**Benchmark results (all configs, same dataset/settings):**

| Configuration | Duration (s) | Req/s | Output tok/s | Total tok/s | Mean TTFT (ms) | Mean TPOT (ms) | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | 24.35 | 3.29 | 841.22 | 1090.87 | 576.17 | 7.28 | N/A |
| Speculative (nspec=3) | 15.92 | 5.02 | **1286.09** | 1667.77 | 26.88 | 5.94 | 15.36% |
| FP8 quantization | 12.65 | 6.32 | **1618.27** | 2098.63 | 21.49 | 4.87 | N/A |
| FP8 + spec (nspec=1) | 16.97 | 4.71 | 1206.84 | 1565.01 | 148.46 | 5.84 | 36.53% |
| FP8 + spec (nspec=2) | 15.95 | 5.02 | **1284.03** | 1665.10 | 67.81 | 5.70 | 22.45% |

**Speculative decoding details:**

| Configuration | Draft tokens per step | Acceptance length | Drafts | Draft tokens | Accepted tokens |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative (nspec=3) | 3 | 1.46 | 13,984 | 41,952 | 6,443 |
| FP8 + spec (nspec=1) | 1 | 1.37 | 14,953 | 14,953 | 5,463 |
| FP8 + spec (nspec=2) | 2 | 1.45 | 14,096 | 28,192 | 6,329 |

**Scoring:** Speculative decoding passes (1286 > 1250 tok/s, 25 pts). FP8 passes (1618 > 1550 tok/s, 10 pts). FP8+speculative does not reach the 1750 tok/s threshold — best result was 1284 tok/s with nspec=2. The draft head's low acceptance rate (~36% at position 0, dropping to ~9% at position 1) means additional draft tokens add more overhead than accepted tokens. Training with more samples (e.g. 10,000+) would likely improve acceptance and close this gap.

---

**Question 1: Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?**

Autoregressive decoding is memory-bandwidth-bound, not compute-bound — the GPU spends most of each step loading model weights from HBM, while compute units sit idle. Speculative decoding exploits this: the draft head produces k candidate tokens cheaply (small model, reuses cached states), and the verifier checks all k tokens in a single forward pass that costs only marginally more than verifying one token (same weight loads, slightly wider activation tensors).

Even at 15% overall acceptance rate, the acceptance *length* of 1.46 means each verification step produces ~1.46 tokens on average instead of exactly 1. That is a 46% increase in tokens per verifier forward pass. The draft overhead (running the small head) is nearly free compared to the verifier's weight-loading cost. So throughput improves from 841 → 1286 tok/s (53% gain) despite most draft tokens being rejected, because each verifier call amortizes its memory-bandwidth cost across more accepted tokens. TPOT dropped from 7.28 to 5.94 ms, confirming faster per-token generation.

**Question 2: How many speculative tokens are optimal for this setup?**

For unquantized speculative decoding, **nspec=3** was optimal, achieving 1286 tok/s with acceptance length 1.46. The per-position acceptance rates (36.2%, 8.6%, 1.3%) show steep decay — position 2 barely contributes. Going to nspec=4+ would add draft overhead with negligible acceptance at position 3.

For FP8+speculative, **nspec=2** was optimal at 1284 tok/s vs. 1207 tok/s for nspec=1. With nspec=1, acceptance length is 1.37 (each step yields 1.37 tokens). With nspec=2, acceptance length rises to 1.45 and TPOT improves (5.70 vs 5.84 ms), showing the second draft position contributes enough accepted tokens to offset its cost. The position-0 acceptance (~36%) is identical across configs, confirming quantization doesn't degrade draft quality at the first position.

The reference results used nspec=2 for unquantized and nspec=1 for FP8+speculative — our setup differs because our draft head has lower acceptance at positions 1-2, making additional draft tokens less rewarding with the FP8 verifier. The FP8 verifier is already fast (4.87 ms TPOT), so the bar for speculative decoding to add value is higher: draft overhead must be justified by proportionally more accepted tokens.

**Main question — Which should be done first: speculative decoding training or quantization?**

Quantization should be done first. The draft head is trained to predict a specific verifier's hidden states and output distribution. If you train the draft head on the BF16 verifier and then quantize the verifier to FP8, the verifier's distribution shifts and the draft head's predictions become less accurate — wasting training effort. Training the draft head against the already-quantized FP8 verifier produces a head that matches the actual deployment distribution. Additionally, FP8 quantization is a one-shot process (no training data needed, runs in minutes), while draft head training is iterative and expensive. It makes sense to finalize the verifier first, then train the draft head to match it.

Speculative decoding benchmark results (nspec=3):
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  15.92     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              5.02      
Output token throughput (tok/s):         1286.09   
Peak output token throughput (tok/s):    926.00    
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1667.77   
---------------Time to First Token----------------
Mean TTFT (ms):                          26.88     
Median TTFT (ms):                        26.04     
P99 TTFT (ms):                           35.29     
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.94      
Median TPOT (ms):                        6.05      
P99 TPOT (ms):                           6.62      
---------------Inter-token Latency----------------
Mean ITL (ms):                           8.67      
Median ITL (ms):                         8.65      
P99 ITL (ms):                            9.43      
---------------Speculative Decoding---------------
Acceptance rate (%):                     15.36     
Acceptance length:                       1.46      
Drafts:                                  13984     
Draft tokens:                            41952     
Accepted tokens:                         6443      
Per-position acceptance (%):
  Position 0:                            36.18     
  Position 1:                            8.59      
  Position 2:                            1.30      
==================================================
```

FP8 quantization benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  12.65     
Total input tokens:                      6078      
Total generated tokens:                  20476     
Request throughput (req/s):              6.32      
Output token throughput (tok/s):         1618.27   
Peak output token throughput (tok/s):    1640.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          2098.63   
---------------Time to First Token----------------
Mean TTFT (ms):                          21.49     
Median TTFT (ms):                        20.93     
P99 TTFT (ms):                           26.14     
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          4.87      
Median TPOT (ms):                        4.87      
P99 TPOT (ms):                           4.90      
---------------Inter-token Latency----------------
Mean ITL (ms):                           4.87      
Median ITL (ms):                         4.87      
P99 ITL (ms):                            5.29      
==================================================
```

FP8 + speculative decoding benchmark results (best: nspec=2):

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  15.95     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              5.02      
Output token throughput (tok/s):         1284.03   
Peak output token throughput (tok/s):    973.00    
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1665.10   
---------------Time to First Token----------------
Mean TTFT (ms):                          67.81     
Median TTFT (ms):                        24.97     
P99 TTFT (ms):                           464.93    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.70      
Median TPOT (ms):                        5.78      
P99 TPOT (ms):                           6.54      
---------------Inter-token Latency----------------
Mean ITL (ms):                           8.25      
Median ITL (ms):                         8.21      
P99 ITL (ms):                            9.10      
---------------Speculative Decoding---------------
Acceptance rate (%):                     22.45     
Acceptance length:                       1.45      
Drafts:                                  14096     
Draft tokens:                            28192     
Accepted tokens:                         6329      
Per-position acceptance (%):
  Position 0:                            36.03     
  Position 1:                            8.87      
==================================================
```

FP8 + speculative decoding benchmark results (nspec=1, for comparison):

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  16.97     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              4.71      
Output token throughput (tok/s):         1206.84   
Peak output token throughput (tok/s):    1016.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1565.01   
---------------Time to First Token----------------
Mean TTFT (ms):                          148.46    
Median TTFT (ms):                        24.56     
P99 TTFT (ms):                           1294.90   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.84      
Median TPOT (ms):                        5.83      
P99 TPOT (ms):                           6.86      
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.97      
Median ITL (ms):                         7.88      
P99 ITL (ms):                            8.82      
---------------Speculative Decoding---------------
Acceptance rate (%):                     36.53     
Acceptance length:                       1.37      
Drafts:                                  14953     
Draft tokens:                            14953     
Accepted tokens:                         5463      
Per-position acceptance (%):
  Position 0:                            36.53     
==================================================
```